<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab%202%20task%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
import random
from collections import deque

import matplotlib
matplotlib.use("Agg")   # saves the plot to a file; remove this line (and plt.show() below) to open a window instead
import matplotlib.pyplot as plt

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}
OPPOSITE = {"Up": "Down", "Down": "Up", "Left": "Right", "Right": "Left"}


def apply_move(state, move):
    b = state.index(0)
    r, c = divmod(b, 3)
    dr, dc = MOVES[move]
    n = (r + dr) * 3 + (c + dc)
    s = list(state)
    s[b], s[n] = s[n], s[b]
    return tuple(s)


def legal_moves(state):
    r, c = divmod(state.index(0), 3)
    return [m for m, (dr, dc) in MOVES.items() if 0 <= r + dr < 3 and 0 <= c + dc < 3]


def successors(state):
    return [apply_move(state, m) for m in legal_moves(state)]


def random_start(goal, n_moves, rng):
    """Apply n random moves to the goal (never undoing the previous move), so the result is solvable."""
    state, last = goal, None
    for _ in range(n_moves):
        options = [m for m in legal_moves(state) if last is None or m != OPPOSITE[last]]
        last = rng.choice(options)
        state = apply_move(state, last)
    return state


def misplaced(state, goal):
    return sum(1 for a, g in zip(state, goal) if a != 0 and a != g)


def manhattan(state, goal):
    pos = {v: i for i, v in enumerate(goal)}
    return sum(abs(i // 3 - pos[v] // 3) + abs(i % 3 - pos[v] % 3)
               for i, v in enumerate(state) if v != 0)


def bfs(start, goal):
    """Returns (solution depth, nodes expanded)."""
    if start == goal:
        return 0, 0
    expanded = 0
    frontier = deque([(start, 0)])
    seen = {start}
    while frontier:
        state, d = frontier.popleft()
        expanded += 1
        for nxt in successors(state):
            if nxt in seen:
                continue
            if nxt == goal:
                return d + 1, expanded
            seen.add(nxt)
            frontier.append((nxt, d + 1))


def astar(start, goal, h):
    """Returns (solution depth, nodes expanded)."""
    expanded, tie = 0, 0
    frontier = [(h(start, goal), 0, tie, start)]
    best_g = {start: 0}
    while frontier:
        f, g, _, state = heapq.heappop(frontier)
        if g > best_g[state]:
            continue
        if state == goal:
            return g, expanded
        expanded += 1
        for nxt in successors(state):
            ng = g + 1
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                tie += 1
                heapq.heappush(frontier, (ng + h(nxt, goal), ng, tie, nxt))


if __name__ == "__main__":
    rng = random.Random(7)              # change the seed for different start states
    walk_lengths = [5, 10, 15, 20, 25]
    rows = []
    for n in walk_lengths:
        start = random_start(GOAL, n, rng)
        depth, e_bfs = bfs(start, GOAL)
        d2, e_mis = astar(start, GOAL, misplaced)
        d3, e_man = astar(start, GOAL, manhattan)
        assert depth == d2 == d3
        rows.append((n, start, depth, e_bfs, e_mis, e_man))

    print(f"{'Random moves':<13}{'Start state':<28}{'Depth':>6}{'BFS':>10}{'A* misplaced':>14}{'A* Manhattan':>14}")
    for n, start, depth, b, m1, m2 in rows:
        print(f"{n:<13}{str(start):<28}{depth:>6}{b:>10}{m1:>14}{m2:>14}")

    depths = [r[2] for r in rows]
    plt.figure(figsize=(8, 5))
    plt.plot(depths, [r[3] for r in rows], "o-", label="BFS")
    plt.plot(depths, [r[4] for r in rows], "s-", label="A* (misplaced tiles)")
    plt.plot(depths, [r[5] for r in rows], "^-", label="A* (Manhattan)")
    plt.yscale("log")
    plt.xlabel("Solution depth")
    plt.ylabel("Nodes expanded (log scale)")
    plt.title("8-puzzle: solution depth vs nodes expanded")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig("depth_vs_nodes.png", dpi=150)
    print("\nSaved plot to depth_vs_nodes.png")
    # plt.show()   # uncomment (and remove matplotlib.use("Agg")) to display the plot in a window

Random moves Start state                  Depth       BFS  A* misplaced  A* Manhattan
5            (2, 0, 3, 1, 4, 6, 7, 5, 8)      5        28             5             5
10           (2, 4, 0, 1, 8, 3, 7, 6, 5)     10       272            35            12
15           (4, 1, 5, 0, 3, 6, 7, 2, 8)     11       505            63            26
20           (1, 6, 5, 3, 2, 8, 4, 7, 0)     16      4914           563           125
25           (6, 5, 3, 0, 2, 7, 8, 1, 4)     25    127346         29246          3464

Saved plot to depth_vs_nodes.png
